# Main Data Loading, Prep, Train, Test, Eval Flow

This notebook wires together the existing `traffic_rl` modules for an end-to-end run.

Flow:
1. Load project configs and data paths
2. Inspect PEMS tensor data
3. Build CityFlow demand splits (train/val/test)
4. Train an agent
5. Evaluate trained vs untrained across splits

Each code cell intentionally uses a different output style (text, JSON, HTML table, artifact paths, ASCII bars).

In [7]:
from __future__ import annotations

import copy
import importlib.util
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import yaml
from IPython.display import HTML, JSON, Markdown, display

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repo root from {start}. Expected pyproject.toml/src/configs in a parent directory."
    )

REPO_ROOT = _find_repo_root(Path.cwd()).resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from traffic_rl.config import load_config
from traffic_rl.evaluation import run_evaluation
from traffic_rl.pems.pipeline import build_cityflow_demands, load_pems_demand_config
from traffic_rl.training import run_training

HAS_CITYFLOW = importlib.util.find_spec("cityflow") is not None
print(f"Working directory: {Path.cwd()}")
print(f"Repository root: {REPO_ROOT}")
print(f"CityFlow available: {HAS_CITYFLOW}")

Working directory: /home/hd/projects/traffic-rl/notebooks
Repository root: /home/hd/projects/traffic-rl
CityFlow available: True


## Step 1: Set Run Controls

The next code cell defines quick-run settings and paths for configs.
- Keep `QUICK_MODE = True` for fast notebook iteration.
- Increase episodes or max steps when you want stronger training signals.
- Paths are resolved from the detected repo root, so this works whether you run from the project root or the notebooks folder.

In [8]:
# Quick-mode controls for a faster notebook run.
QUICK_MODE = True
TRAIN_EPISODES = 3
TRAIN_MAX_STEPS = 40
EVAL_EPISODES = 3

PEMS_CONFIG_PATH = REPO_ROOT / "configs" / "pems04_to_cityflow.example.yaml"
CITYFLOW_BASE_CONFIG_PATH = REPO_ROOT / "configs" / "cityflow.quick.yaml"
MOCK_BASE_CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"

path_status = {
    "quick_mode": QUICK_MODE,
    "paths": {
        "pems_config": {"path": str(PEMS_CONFIG_PATH), "exists": PEMS_CONFIG_PATH.exists()},
        "cityflow_base_config": {"path": str(CITYFLOW_BASE_CONFIG_PATH), "exists": CITYFLOW_BASE_CONFIG_PATH.exists()},
        "mock_base_config": {"path": str(MOCK_BASE_CONFIG_PATH), "exists": MOCK_BASE_CONFIG_PATH.exists()},
    },
}
display(JSON(path_status, expanded=True))

<IPython.core.display.JSON object>

## Step 2: Inspect Raw PEMS Tensor

The next cell loads the PEMS NPZ file and prints distribution stats.
This is a sanity check before generating any CityFlow demand files.
If shape or value ranges look unexpected, stop here and verify the source data/config first.

In [9]:
pems_cfg = load_pems_demand_config(PEMS_CONFIG_PATH)
pems_npz = np.load(pems_cfg.pems_npz_path)
tensor = pems_npz["data"]

stats = {
    "shape": tuple(int(v) for v in tensor.shape),
    "dtype": str(tensor.dtype),
    "min": float(tensor.min()),
    "mean": float(tensor.mean()),
    "max": float(tensor.max()),
}

flow_feature_idx = pems_cfg.flow_feature_index
sensor_rows = []
for sensor_idx in pems_cfg.sensor_indices[:4]:
    values = tensor[:, sensor_idx, flow_feature_idx]
    sensor_rows.append(
        f"<tr><td>{sensor_idx}</td><td>{float(values.min()):.2f}</td><td>{float(values.mean()):.2f}</td><td>{float(values.max()):.2f}</td></tr>"
    )

table_html = ""
table_html += "<h3>PEMS Data Snapshot (HTML Table Output)</h3>"
table_html += "<p>Tensor shape: {} | dtype: {}</p>".format(stats["shape"], stats["dtype"])
table_html += "<table border='1' cellpadding='6'><tr><th>Sensor</th><th>Min</th><th>Mean</th><th>Max</th></tr>"
table_html += "".join(sensor_rows)
table_html += "</table>"

display(HTML(table_html))
print("Global stats:", stats)

Sensor,Min,Mean,Max
0,0.00,244.29,626.00
1,0.00,196.51,536.00
2,0.00,239.96,660.00
3,0.00,187.08,507.00


Global stats: {'shape': (16992, 307, 3), 'dtype': 'float64', 'min': 0.0, 'mean': 91.74140778613855, 'max': 919.0}


## Step 3: Build CityFlow Demand Splits

This step runs your existing demand pipeline to create train/val/test flow files.
Quick mode reduces sensors and split sizes so the notebook stays responsive.
The output summary shows generated vehicle counts and split metadata.

In [10]:
prep_cfg = copy.deepcopy(pems_cfg)

if QUICK_MODE:
    if len(prep_cfg.sensor_indices) > 2:
        prep_cfg.sensor_indices = prep_cfg.sensor_indices[:2]
    prep_cfg.split.train_days = 1
    prep_cfg.split.val_days = 1
    prep_cfg.split.test_days = 1
    prep_cfg.arrival_process = "uniform"

prep_output_dir = REPO_ROOT / "outputs" / ("pems04_notebook_quick" if QUICK_MODE else "pems04_notebook_full")
prep_cfg.output_dir = str(prep_output_dir)

prep_outputs = build_cityflow_demands(prep_cfg)
prep_summary = json.loads(Path(prep_outputs.summary_file).read_text(encoding="utf-8"))

display(Markdown("### Demand Build Artifacts"))
print(f"train flow : {prep_outputs.train_flow_file}")
print(f"val flow   : {prep_outputs.val_flow_file}")
print(f"test flow  : {prep_outputs.test_flow_file}")
print(f"summary    : {prep_outputs.summary_file}")
display(JSON(prep_summary, expanded=True))

Building test split: 100%|██████████| 288/288 [00:00<00:00, 1217.03window/s]


### Demand Build Artifacts

train flow : /home/hd/projects/traffic-rl/outputs/pems04_notebook_quick/flow_train.json
val flow   : /home/hd/projects/traffic-rl/outputs/pems04_notebook_quick/flow_val.json
test flow  : /home/hd/projects/traffic-rl/outputs/pems04_notebook_quick/flow_test.json
summary    : /home/hd/projects/traffic-rl/outputs/pems04_notebook_quick/summary.json


<IPython.core.display.JSON object>

## Step 4: Preview Generated Demand Entries

The preview cell reads a small sample from each split file.
Use it to verify route assignment and timing structure before training.
If needed, adjust route probabilities in the PEMS config and rebuild.

In [11]:
def _load_preview(path: Path, limit: int = 3):
    rows = json.loads(path.read_text(encoding="utf-8"))
    return rows[:limit], len(rows)

train_preview, train_count = _load_preview(Path(prep_outputs.train_flow_file))
val_preview, val_count = _load_preview(Path(prep_outputs.val_flow_file))
test_preview, test_count = _load_preview(Path(prep_outputs.test_flow_file))

print("Vehicle counts by split:")
print(f"  train: {train_count}")
print(f"  val  : {val_count}")
print(f"  test : {test_count}")

display(JSON({
    "train_preview": train_preview,
    "val_preview": val_preview,
    "test_preview": test_preview,
}, expanded=False))

Vehicle counts by split:
  train: 93213
  val  : 126706
  test : 129913


<IPython.core.display.JSON object>

In [12]:
def _resolve_from_yaml(config_path: Path, value: str) -> str:
    path = Path(value)
    if path.is_absolute():
        return str(path)
    return str((config_path.parent / path).resolve())


def _to_engine_relative(engine_dir: Path, flow_path: Path) -> str:
    return os.path.relpath(flow_path.resolve(), engine_dir.resolve())


split_cfg_paths: dict[str, str] = {}
split_mode = "mock"

if HAS_CITYFLOW and CITYFLOW_BASE_CONFIG_PATH.exists():
    split_mode = "cityflow"
    base_raw = yaml.safe_load(CITYFLOW_BASE_CONFIG_PATH.read_text(encoding="utf-8")) or {}
    base_engine_path = Path(_resolve_from_yaml(CITYFLOW_BASE_CONFIG_PATH, str(base_raw["env"]["cityflow_config_path"])))
    engine_raw = json.loads(base_engine_path.read_text(encoding="utf-8"))

    notebook_cfg_dir = REPO_ROOT / "outputs" / "notebook_run"
    notebook_cfg_dir.mkdir(parents=True, exist_ok=True)

    split_to_flow = {
        "train": Path(prep_outputs.train_flow_file),
        "val": Path(prep_outputs.val_flow_file),
        "test": Path(prep_outputs.test_flow_file),
    }

    for split_name, flow_path in split_to_flow.items():
        split_engine = copy.deepcopy(engine_raw)
        engine_dir = Path(split_engine["dir"]).resolve()
        split_engine["flowFile"] = _to_engine_relative(engine_dir, flow_path)

        engine_out = notebook_cfg_dir / f"cityflow_engine_{split_name}.json"
        engine_out.write_text(json.dumps(split_engine, indent=2), encoding="utf-8")

        split_cfg_raw = copy.deepcopy(base_raw)
        split_cfg_raw["env"]["cityflow_config_path"] = str(engine_out)
        split_cfg_path = notebook_cfg_dir / f"cityflow_{split_name}.yaml"
        split_cfg_path.write_text(yaml.safe_dump(split_cfg_raw, sort_keys=False), encoding="utf-8")
        split_cfg_paths[split_name] = str(split_cfg_path)
else:
    split_cfg_paths = {
        "train": str(MOCK_BASE_CONFIG_PATH),
        "val": str(MOCK_BASE_CONFIG_PATH),
        "test": str(MOCK_BASE_CONFIG_PATH),
    }

display(JSON({
    "split_mode": split_mode,
    "split_config_paths": split_cfg_paths,
}, expanded=True))

<IPython.core.display.JSON object>

## Step 5: Create Split-Specific Runtime Configs

This cell prepares per-split config files used for training and evaluation.
If CityFlow is available, it rewrites the engine `flowFile` for each split.
If not, it falls back to mock backend configs so the notebook still runs end-to-end.

In [13]:
train_cfg = load_config(split_cfg_paths["train"])
train_cfg.training.episodes = TRAIN_EPISODES
train_cfg.training.max_steps = TRAIN_MAX_STEPS
train_cfg.output_dir = str(REPO_ROOT / "outputs" / "notebook_run")

train_summary = run_training(train_cfg)

def _bar(value: float, scale: float = 3.0, width: int = 30) -> str:
    clipped = max(-width, min(width, int(round(value / scale))))
    if clipped >= 0:
        return "+" * clipped
    return "-" * abs(clipped)

print(f"Training episodes: {train_summary.episodes}")
print(f"Average reward : {train_summary.average_reward:.3f}")
print("Episode rewards (ASCII bars):")
for idx, reward in enumerate(train_summary.episode_rewards, start=1):
    print(f"  ep {idx:02d} | {reward:8.3f} | {_bar(reward)}")

Training episodes: 100%|██████████| 3/3 [00:01<00:00,  2.58ep/s]

Training episodes: 3
Average reward : -181.667
Episode rewards (ASCII bars):
  ep 01 | -261.000 | ------------------------------
  ep 02 |  -89.000 | ------------------------------
  ep 03 | -195.000 | ------------------------------


## Step 6: Train Agent on Train Split

Training uses your current `TRAIN_EPISODES` and `TRAIN_MAX_STEPS` settings.
The ASCII bars provide a quick visual of per-episode reward behavior.
A backend-specific checkpoint is saved under `outputs/notebook_run`.

In [14]:
eval_rows = []

for split_name in ("train", "val", "test"):
    eval_cfg = load_config(split_cfg_paths[split_name])
    eval_cfg.training.max_steps = TRAIN_MAX_STEPS
    eval_cfg.output_dir = train_cfg.output_dir

    trained = run_evaluation(
        eval_cfg,
        episodes=EVAL_EPISODES,
        checkpoint_path=None,
        replay_file=None,
        load_checkpoint=True,
        show_progress=False,
    )
    untrained = run_evaluation(
        eval_cfg,
        episodes=EVAL_EPISODES,
        checkpoint_path=None,
        replay_file=None,
        load_checkpoint=False,
        show_progress=False,
    )

    eval_rows.append({
        "split": split_name,
        "trained_avg_reward": float(trained.average_reward),
        "untrained_avg_reward": float(untrained.average_reward),
        "delta": float(trained.average_reward - untrained.average_reward),
        "trained_avg_queue": float(trained.average_queue),
        "trained_avg_throughput": float(trained.average_throughput),
    })

html = ""
html += "<h3>Evaluation Summary (HTML Table)</h3>"
html += "<table border='1' cellpadding='6'>"
html += "<tr><th>Split</th><th>Trained Avg Reward</th><th>Untrained Avg Reward</th><th>Delta</th><th>Trained Avg Queue</th><th>Trained Avg Throughput</th></tr>"
for row in eval_rows:
    html += (
        f"<tr><td>{row['split']}</td><td>{row['trained_avg_reward']:.3f}</td><td>{row['untrained_avg_reward']:.3f}</td>"
        f"<td>{row['delta']:.3f}</td><td>{row['trained_avg_queue']:.3f}</td><td>{row['trained_avg_throughput']:.3f}</td></tr>"
    )
html += "</table>"
display(HTML(html))
eval_rows

Split,Trained Avg Reward,Untrained Avg Reward,Delta,Trained Avg Queue,Trained Avg Throughput
train,-186.000,-698.000,512.000,0.291,28.600
val,-140.000,-1104.000,964.000,0.219,44.700
test,-76.000,-548.000,472.000,0.119,24.850


[{'split': 'train',
  'trained_avg_reward': -186.0,
  'untrained_avg_reward': -698.0,
  'delta': 512.0,
  'trained_avg_queue': 0.290625,
  'trained_avg_throughput': 28.6},
 {'split': 'val',
  'trained_avg_reward': -140.0,
  'untrained_avg_reward': -1104.0,
  'delta': 964.0,
  'trained_avg_queue': 0.21875,
  'trained_avg_throughput': 44.7},
 {'split': 'test',
  'trained_avg_reward': -76.0,
  'untrained_avg_reward': -548.0,
  'delta': 472.0,
  'trained_avg_queue': 0.11875,
  'trained_avg_throughput': 24.85}]

## Step 7: Evaluate Trained vs Untrained and Save Report

The evaluation cell compares trained and untrained policies on train/val/test splits.
Focus on `delta` first, then queue and throughput metrics for operational context.
The final cell persists a compact JSON report for downstream analysis or plotting.

In [15]:
report = {
    "meta": {
        "quick_mode": QUICK_MODE,
        "split_mode": split_mode,
        "train_episodes": TRAIN_EPISODES,
        "eval_episodes": EVAL_EPISODES,
        "train_max_steps": TRAIN_MAX_STEPS,
    },
    "prep_summary": prep_summary,
    "training": asdict(train_summary),
    "evaluation": eval_rows,
}

report_path = Path(train_cfg.output_dir) / "notebook_flow_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

display(Markdown(f"### Saved notebook report\n`{report_path}`"))
print(json.dumps(report["meta"], indent=2))

### Saved notebook report
`/home/hd/projects/traffic-rl/outputs/notebook_run/notebook_flow_report.json`

{
  "quick_mode": true,
  "split_mode": "cityflow",
  "train_episodes": 3,
  "eval_episodes": 3,
  "train_max_steps": 40
}
